## Libraries

In [1]:
import math
import random
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from torchinfo import summary
from transformers import ConvNextV2ForImageClassification

from lib.muon import Muon
from src.utils import query_gallery_split
from src.metrics import evaluate

## Config

In [2]:
torch._dynamo.config.recompile_limit = 16

In [3]:
seed = 56
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Datasets

In [5]:
class CarsDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, data, transform=None):
        self.root_path = root_path
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_path = self.root_path / self.data.loc[index, "image_path"]
        assert img_path.is_file()
        try:
            image = torchvision.io.read_image(img_path)
        except RuntimeError:
            image = Image.open(img_path)
            image = transforms.functional.pil_to_tensor(image)

        x1, y1, x2, y2 = self.data.loc[index, ["x1", "y1", "x2", "y2"]]
        if 0 <= x1 < x2 <= image.shape[2] and 0 <= y1 < y2 <= image.shape[1]:
            image = image[:, y1:y2, x1:x2]

        if image.shape[0] == 4:
            image = image[:3]
        elif image.shape[0] == 1:
            image = image.repeat(3, 1, 1)

        if self.transform is not None:
            image = self.transform(image)

        return image, self.data.loc[index, "class"], self.data.loc[index, "weight"]

In [6]:
class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, data, transform=None):
        self.root_path = root_path
        self.data = data[["image_id", "x", "y", "w", "h"]].to_numpy()
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data[index]

        img_path = (self.root_path / row[0]).with_suffix(".jpg")
        assert img_path.is_file()
        image = torchvision.io.read_image(img_path)

        x, y, w, h = row[1:]
        image = image[:, y : y + h, x : x + w]
        assert image.shape == (3, h, w)

        if self.transform is not None:
            image = self.transform(image)

        return image

## Data

In [7]:
data_1 = pd.read_csv("./CompCars/boxes.csv")
data_1["class"] = data_1["image_path"].str.split("/").str.slice(0, -1).str.join("/")
data_1["image_path"] = "CompCars/data/image/" + data_1["image_path"]

data_2 = pd.read_csv("./Car-1000/boxes.csv")
data_2["class"] = data_2["image_path"].str.split("/").str.slice(0, -1).str.join("/")
data_2["image_path"] = "Car-1000/all_images/" + data_2["image_path"]

train_data = pd.concat([data_1, data_2], ignore_index=True)

class2id = {}
for idx, cls in enumerate(train_data["class"].unique()):
    class2id[cls] = idx
train_data["class"] = train_data["class"].map(class2id)

train_data["weight"] = (
    1 / np.sqrt(train_data.groupby("class")["class"].transform("count"))
)
train_data["weight"] = (
    train_data["weight"] / train_data["weight"].mean()
).astype(np.float32)

train_data

,image_path,x1,y1,x2,y2,class,weight
0,CompCars/data/image/1/1101/2011/07b90decb92ba6...,124,33,674,480,0,1.905164
1,CompCars/data/image/1/1101/2011/2272c7d324cf79...,52,153,849,506,0,1.905164
2,CompCars/data/image/1/1101/2011/3a62131af5fe8e...,85,107,682,469,0,1.905164
3,CompCars/data/image/1/1101/2011/5133ca181b82af...,91,23,759,551,0,1.905164
4,CompCars/data/image/1/1101/2011/6632ec1c2f7f87...,58,134,707,504,0,1.905164
...,...,...,...,...,...,...,...
276988,Car-1000/all_images/0999/0999_0170.jpg,78,10,572,452,5445,2.087003
276989,Car-1000/all_images/0999/0999_0406.jpg,77,25,501,342,5445,2.087003
276990,Car-1000/all_images/0999/0999_0466.jpg,98,56,386,315,5445,2.087003
276991,Car-1000/all_images/0999/0999_0467.jpg,23,91,455,313,5445,2.087003


In [8]:
data = pd.read_csv("./data/train.csv")
data

,image_id,x,y,w,h,vehicle_id,camera_id
0,f0d68d754a8a444b8b49628b0633068c,1199,491,619,589,403,90
1,b94be87b5ced41f5868f2dfb415100e4,0,308,437,420,403,89
2,296ab330ff8e49dd878d922c10352ca0,1199,433,682,594,403,90
3,5b7baa07e07640449243f4619a3077cc,1,317,434,414,403,89
4,ee3240d083594b5dbed8d8d28c236f91,0,332,436,395,403,89
...,...,...,...,...,...,...,...
9551,33c775ad393847f180aa715ac2d4c970,631,201,822,626,330,48
9552,56b0fd4da91545038e7228e5fe3706f0,630,201,823,624,330,48
9553,24ab438a002044ceaf3ab0cfafb811e9,1014,35,713,516,330,73
9554,f15c0a764e054d5ebf103925ce5f13ec,1016,33,707,518,330,73


In [9]:
query_df, gallery_df = query_gallery_split(data)
len(query_df), len(gallery_df)

(5560, 3996)

In [10]:
train_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224), antialias=True),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        transforms.RandomErasing(),
    ]
)
train_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
    ConvertImageDtype()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False)
)

In [11]:
train_dataset = CarsDataset(Path(""), train_data, train_transforms)
train_dataset[0]

(tensor([[[ 2.2489,  2.2489,  2.2489,  ...,  0.4508,  0.3994,  0.3138],
          [ 2.2489,  2.2489,  2.2489,  ...,  0.4851,  0.4337,  0.3481],
          [ 2.2489,  2.2489,  2.2489,  ...,  0.4851,  0.4508,  0.3823],
          ...,
          [ 1.6838,  1.7352,  1.7180,  ..., -1.3815, -1.3130, -1.3815],
          [ 1.6495,  1.7352,  1.6838,  ..., -1.3815, -1.3987, -1.3473],
          [ 1.6495,  1.6838,  1.5982,  ..., -1.4329, -1.3815, -1.3473]],
 
         [[ 2.4286,  2.4286,  2.4286,  ...,  0.9055,  0.8529,  0.7829],
          [ 2.4286,  2.4286,  2.4286,  ...,  0.9230,  0.8704,  0.8179],
          [ 2.4286,  2.4286,  2.4286,  ...,  0.9230,  0.9055,  0.8529],
          ...,
          [ 2.3585,  2.4111,  2.3761,  ..., -1.0553, -1.0028, -1.0553],
          [ 2.3235,  2.4111,  2.3585,  ..., -1.0553, -1.0728, -1.0203],
          [ 2.3235,  2.3585,  2.2710,  ..., -1.1078, -1.0553, -1.0203]],
 
         [[ 2.6400,  2.6400,  2.6400,  ...,  1.2805,  1.2282,  1.2282],
          [ 2.6400,  2.6400,

In [12]:
val_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224), antialias=True),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)
val_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ConvertImageDtype()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)

In [13]:
val_dataset = torch.utils.data.ConcatDataset(
    [
        InferenceDataset(Path("data/images/"), query_df, val_transforms),
        InferenceDataset(Path("data/images/"), gallery_df, val_transforms),
    ]
)
val_dataset[0]

tensor([[[-1.2445, -1.2788, -1.3130,  ...,  0.1597,  0.8276,  0.8104],
         [-1.1075, -1.0904, -1.1589,  ..., -0.0116,  0.1426,  0.1597],
         [ 0.4508,  0.5878,  0.6906,  ...,  0.0569,  0.0398,  0.0398],
         ...,
         [ 0.1254, -0.3369, -0.5424,  ...,  0.1083,  0.0741,  0.0569],
         [ 0.1254, -0.2513, -0.5424,  ...,  0.0912,  0.0741,  0.1254],
         [ 0.1083, -0.1999, -0.5596,  ...,  0.0569,  0.0912,  0.0912]],

        [[-1.1253, -1.1604, -1.1954,  ...,  0.3452,  1.0280,  1.0105],
         [-0.9853, -0.9678, -1.0378,  ...,  0.1702,  0.3277,  0.3452],
         [ 0.6078,  0.7479,  0.8529,  ...,  0.2402,  0.2227,  0.2227],
         ...,
         [ 0.2927, -0.1275, -0.3550,  ...,  0.2577,  0.2227,  0.2052],
         [ 0.2927, -0.0399, -0.3550,  ...,  0.2402,  0.2227,  0.2752],
         [ 0.2927,  0.0126, -0.3550,  ...,  0.2052,  0.2402,  0.2402]],

        [[-0.8284, -0.8633, -0.8981,  ...,  0.6531,  1.3328,  1.3154],
         [-0.6890, -0.6715, -0.7413,  ...,  0

## Dataloaders

In [14]:
train_dataloader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=32,
    num_workers=8,
    drop_last=True,
    shuffle=True,
    pin_memory=True,
    persistent_workers=True,
)

In [15]:
val_dataloader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=32,
    num_workers=8,
    drop_last=False,
    shuffle=False,
    pin_memory=True,
    persistent_workers=True,
)

## Model

In [16]:
class Model(nn.Module):
    def __init__(self, num_classes):
        super(Model, self).__init__()
        self.net = ConvNextV2ForImageClassification.from_pretrained(
            "facebook/convnextv2-base-22k-224", drop_path_rate=0.5
        )
        self.net.classifier = nn.Identity()
        self.fc = nn.Linear(1024, 512, bias=False)
        self.head = nn.Linear(512, num_classes, bias=False)
        self.s = math.sqrt(2) * math.log(num_classes - 1)

    def forward(self, x):
        embeds = self.fc(self.net(pixel_values=x).logits)
        embeds = F.normalize(embeds, p=2, dim=-1)
        weight = F.normalize(self.head.weight, p=2, dim=-1)
        logits = F.linear(embeds * self.s, weight)
        logits = logits.float()
        return embeds, logits

In [17]:
model = Model(len(class2id))
model.to(device)

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

Model(
  (net): ConvNextV2ForImageClassification(
    (convnextv2): ConvNextV2Model(
      (embeddings): ConvNextV2Embeddings(
        (patch_embeddings): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
        (layernorm): ConvNextV2LayerNorm((128,), eps=1e-06, elementwise_affine=True)
      )
      (encoder): ConvNextV2Encoder(
        (stages): ModuleList(
          (0): ConvNextV2Stage(
            (downsampling_layer): ModuleList()
            (layers): ModuleList(
              (0): ConvNextV2Layer(
                (dwconv): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
                (layernorm): ConvNextV2LayerNorm((128,), eps=1e-06, elementwise_affine=True)
                (pwconv1): Linear(in_features=128, out_features=512, bias=True)
                (act): GELUActivation()
                (grn): ConvNextV2GRN()
                (pwconv2): Linear(in_features=512, out_features=128, bias=True)
                (drop_path): Identity()
         

In [18]:
summary(model, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                                                 Output Shape              Param #
Model                                                                  [1, 512]                  2,788,352
├─ConvNextV2ForImageClassification: 1-1                                [1, 1024]                 --
│    └─ConvNextV2Model: 2-1                                            [1, 1024]                 --
│    │    └─ConvNextV2Embeddings: 3-1                                  [1, 128, 56, 56]          6,528
│    │    └─ConvNextV2Encoder: 3-2                                     [1, 1024, 7, 7]           87,684,224
│    │    └─LayerNorm: 3-3                                             [1, 1024]                 2,048
│    └─Identity: 2-2                                                   [1, 1024]                 --
├─Linear: 1-2                                                          [1, 512]                  524,288
Total params: 91,005,440
Trainable params: 91,005,440
Non-trainable p

In [19]:
model.compile(mode="reduce-overhead", dynamic=False, fullgraph=True)

## Optimizers

In [20]:
muon_params = [
    p
    for name, p in model.named_parameters()
    if name.startswith("net.")
    and (
        name.rstrip("weight").endswith(
            ("pwconv1.", "pwconv2.", "downsampling_layer.1.")
        )
        and not print(name)
    )
]

adam_params = [
    p
    for name, p in model.named_parameters()
    if name.startswith("net.")
    and not (
        name.rstrip("weight").endswith(
            ("pwconv1.", "pwconv2.", "downsampling_layer.1.")
        )
    )
]

optimizer = Muon(
    [
        {"params": muon_params, "lr": 1e-4, "use_muon": True},
        {"params": adam_params, "lr": 1e-4, "use_muon": False},
        {"params": model.fc.weight, "lr": 1e-3, "use_muon": True},
        {"params": model.head.weight, "lr": 1e-3, "use_muon": False},
    ]
)

net.convnextv2.encoder.stages.0.layers.0.pwconv1.weight
net.convnextv2.encoder.stages.0.layers.0.pwconv2.weight
net.convnextv2.encoder.stages.0.layers.1.pwconv1.weight
net.convnextv2.encoder.stages.0.layers.1.pwconv2.weight
net.convnextv2.encoder.stages.0.layers.2.pwconv1.weight
net.convnextv2.encoder.stages.0.layers.2.pwconv2.weight
net.convnextv2.encoder.stages.1.downsampling_layer.1.weight
net.convnextv2.encoder.stages.1.layers.0.pwconv1.weight
net.convnextv2.encoder.stages.1.layers.0.pwconv2.weight
net.convnextv2.encoder.stages.1.layers.1.pwconv1.weight
net.convnextv2.encoder.stages.1.layers.1.pwconv2.weight
net.convnextv2.encoder.stages.1.layers.2.pwconv1.weight
net.convnextv2.encoder.stages.1.layers.2.pwconv2.weight
net.convnextv2.encoder.stages.2.downsampling_layer.1.weight
net.convnextv2.encoder.stages.2.layers.0.pwconv1.weight
net.convnextv2.encoder.stages.2.layers.0.pwconv2.weight
net.convnextv2.encoder.stages.2.layers.1.pwconv1.weight
net.convnextv2.encoder.stages.2.layers.1

## Training

In [21]:
WARMUP_EPOCHS = 1
EPOCHS = 7

In [22]:
main_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1.0,
    end_factor=0.0,
    total_iters=EPOCHS * len(train_dataloader),
)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS * len(train_dataloader),
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, main_scheduler],
    milestones=[WARMUP_EPOCHS * len(train_dataloader)],
)

In [ ]:
for epoch in tqdm(range(WARMUP_EPOCHS + EPOCHS)):
    print(f"EPOCH {epoch}:")

    model.train()

    running_loss = 0.0

    tbar = tqdm(total=len(train_dataloader))
    for x, labels, weights in train_dataloader:
        x = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            _, logits = model(x)
        loss = (F.cross_entropy(logits, labels, reduction="none") * weights).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        running_loss += loss.detach()
        tbar.update(1)

    tbar.close()

    running_loss /= len(train_dataloader)
    print(f"Train Loss: {running_loss.item()}")

    model.eval()

    y_pred = []

    with torch.inference_mode():
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            for x in val_dataloader:
                x = x.to(device, non_blocking=True)
                embeds, _ = model(x)
                y_pred.append(embeds.clone())

    y_pred = torch.cat(y_pred).cpu()

    query_embeds = y_pred[: len(query_df)]
    gallery_embeds = y_pred[-len(gallery_df) :]

    s = (
        F.normalize(query_embeds, p=2, dim=-1)
        @ F.normalize(gallery_embeds, p=2, dim=-1).T
    ).numpy()
    print("Val mAP:", evaluate(query_df, gallery_df, s))

## Save

In [24]:
model.net.save_pretrained("convnextv2-base-pretrained/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Evaluation

In [25]:
from src.utils import train_val_split

In [26]:
train_df, val_df = train_val_split(data, val_size=1/3)
len(train_df), len(val_df)

(6367, 3189)

In [27]:
query_df, gallery_df = query_gallery_split(val_df)
len(query_df), len(gallery_df)

(1862, 1327)

In [28]:
val_dataset = torch.utils.data.ConcatDataset([
    InferenceDataset(Path("data/images/"), query_df, val_transforms),
    InferenceDataset(Path("data/images/"), gallery_df, val_transforms),
])
val_dataset[0]

tensor([[[-0.7650, -0.7993, -0.9020,  ...,  0.2624,  0.2453,  0.0912],
         [-0.6452, -0.8164, -0.7993,  ...,  0.1939,  0.5193,  0.6221],
         [-0.3712, -0.6965, -0.7308,  ...,  0.5022,  0.5022,  0.6221],
         ...,
         [ 1.2385,  1.2043,  1.1358,  ...,  0.3823,  0.3309,  0.4508],
         [ 1.2043,  1.1529,  1.0673,  ...,  0.3309,  0.3823,  0.4679],
         [ 1.1872,  1.1700,  1.1529,  ...,  0.3138,  0.4166,  0.4851]],

        [[-0.6176, -0.6352, -0.7402,  ...,  1.0105,  0.9930,  0.8529],
         [-0.4951, -0.6527, -0.6527,  ...,  0.9405,  1.2731,  1.3957],
         [-0.2150, -0.5651, -0.6001,  ...,  1.2381,  1.2556,  1.3606],
         ...,
         [ 1.3957,  1.3431,  1.2906,  ...,  0.5378,  0.4853,  0.6078],
         [ 1.3431,  1.3081,  1.2031,  ...,  0.4678,  0.5203,  0.6078],
         [ 1.3431,  1.3256,  1.2906,  ...,  0.4503,  0.5553,  0.6254]],

        [[-0.2707, -0.2881, -0.4101,  ..., -0.2358, -0.2881, -0.4798],
         [-0.1835, -0.3404, -0.3404,  ..., -0

In [29]:
val_dataloader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=32,
    num_workers=8,
    drop_last=False,
    shuffle=False,
    pin_memory=True,
    persistent_workers=True,
)

In [30]:
model.eval()

y_pred = []

with torch.inference_mode():
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for x in val_dataloader:
            x = x.to(device, non_blocking=True)
            embeds, _ = model(x)
            y_pred.append(embeds.clone())

y_pred = torch.cat(y_pred).cpu()

query_embeds = y_pred[: len(query_df)]
gallery_embeds = y_pred[-len(gallery_df) :]

s = (
    F.normalize(query_embeds, p=2, dim=-1)
    @ F.normalize(gallery_embeds, p=2, dim=-1).T
).numpy()
print("Val mAP:", evaluate(query_df, gallery_df, s))

Val mAP: 0.5254119763389066
